In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances
import tkinter as tk
from tkinter import ttk, messagebox
import numpy as np
import os

# Check if dataset exists
dataset_path = 'dataset.csv'
if not os.path.exists(dataset_path):
    print(f"Dataset not found at {dataset_path}")
    print("Please ensure the dataset file exists or update the path.")
    exit()

# Load dataset
try:
    df = pd.read_csv(dataset_path)
    print(f"Original dataset size: {len(df)} songs")
    df = df.drop_duplicates(subset=['track_name', 'artists'])
    print(f"Dataset after removing duplicates: {len(df)} songs")
    print(f"Unique artists: {df['artists'].nunique()}")
    print(f"Unique genres: {df['track_genre'].nunique()}")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit()

# All available features for recommendation
ALL_FEATURES = ['danceability', 'energy', 'speechiness', 'acousticness',
                'instrumentalness', 'liveness', 'valence', 'tempo',
                'loudness', 'duration_ms', 'key', 'mode', 'time_signature']

# Check which features exist in dataset
available_features = [f for f in ALL_FEATURES if f in df.columns]
missing_features = [f for f in ALL_FEATURES if f not in df.columns]

print(f"Available features: {len(available_features)}")
print(f"Missing features: {missing_features}")

if len(available_features) < 3:
    print("Error: Need at least 3 features for recommendations")
    exit()

# Global variables
current_features = available_features.copy()
scaler = None
knn = None
X_scaled = None

class ModernButton:
    """Custom modern button class with gradient effects and animations"""
    
    def __init__(self, parent, text, command, bg_color, hover_color, text_color="white", 
                 font=('Arial', 11, 'bold'), width=None, height=None, emoji=""):
        self.parent = parent
        self.text = text
        self.command = command
        self.bg_color = bg_color
        self.hover_color = hover_color
        self.text_color = text_color
        self.font = font
        self.emoji = emoji
        self.is_hovered = False
        
        # Create the button
        self.button = tk.Button(
            parent,
            text=f"{emoji} {text}".strip(),
            command=self.on_click,
            bg=bg_color,
            fg=text_color,
            font=font,
            relief='flat',
            borderwidth=0,
            padx=15,
            pady=10,
            cursor='hand2',
            activebackground=hover_color,
            activeforeground=text_color,
            width=width,
            height=height
        )
        
        # Bind hover events
        self.button.bind("<Enter>", self.on_enter)
        self.button.bind("<Leave>", self.on_leave)
        self.button.bind("<Button-1>", self.on_press)
        self.button.bind("<ButtonRelease-1>", self.on_release)
        
    def on_click(self):
        if self.command:
            self.command()
    
    def on_enter(self, event):
        self.is_hovered = True
        self.button.config(
            bg=self.hover_color,
            relief='raised',
            borderwidth=2
        )
        
    def on_leave(self, event):
        self.is_hovered = False
        self.button.config(
            bg=self.bg_color,
            relief='flat',
            borderwidth=0
        )
        
    def on_press(self, event):
        self.button.config(relief='sunken', borderwidth=2)
        
    def on_release(self, event):
        if self.is_hovered:
            self.button.config(relief='raised', borderwidth=2)
        else:
            self.button.config(relief='flat', borderwidth=0)
    
    def pack(self, **kwargs):
        return self.button.pack(**kwargs)
    
    def grid(self, **kwargs):
        return self.button.grid(**kwargs)
    
    def place(self, **kwargs):
        return self.button.place(**kwargs)

class CustomCheckbox:
    """Custom checkbox with modern styling"""
    
    def __init__(self, parent, text, variable, bg_color="#ffffff", check_color="#4CAF50", 
                 text_color="#333333", font=('Arial', 10)):
        self.parent = parent
        self.text = text
        self.variable = variable
        self.bg_color = bg_color
        self.check_color = check_color
        self.text_color = text_color
        self.font = font
        
        # Create frame for checkbox
        self.frame = tk.Frame(parent, bg=bg_color)
        
        # Create custom checkbox canvas
        self.canvas = tk.Canvas(self.frame, width=20, height=20, bg=bg_color, 
                               highlightthickness=0, relief='flat')
        self.canvas.pack(side='left', padx=(0, 8), pady=2)
        
        # Create label
        self.label = tk.Label(self.frame, text=text, bg=bg_color, fg=text_color, 
                             font=font, cursor='hand2')
        self.label.pack(side='left', pady=2)
        
        # Bind events
        self.canvas.bind("<Button-1>", self.toggle)
        self.label.bind("<Button-1>", self.toggle)
        self.canvas.bind("<Enter>", self.on_enter)
        self.canvas.bind("<Leave>", self.on_leave)
        self.label.bind("<Enter>", self.on_enter)
        self.label.bind("<Leave>", self.on_leave)
        
        # Initial draw
        self.draw_checkbox()
        
        # Trace variable changes
        self.variable.trace('w', self.on_variable_change)
        
    def draw_checkbox(self):
        self.canvas.delete("all")
        
        # Draw checkbox border
        if self.variable.get():
            # Checked state
            self.canvas.create_rectangle(2, 2, 18, 18, fill=self.check_color, 
                                       outline=self.check_color, width=2)
            # Draw checkmark
            self.canvas.create_line(6, 10, 9, 13, fill="white", width=3, capstyle='round')
            self.canvas.create_line(9, 13, 14, 8, fill="white", width=3, capstyle='round')
        else:
            # Unchecked state
            self.canvas.create_rectangle(2, 2, 18, 18, fill="white", 
                                       outline="#CCCCCC", width=2)
    
    def toggle(self, event=None):
        self.variable.set(not self.variable.get())
        
    def on_enter(self, event):
        if not self.variable.get():
            self.canvas.create_rectangle(2, 2, 18, 18, fill="#f0f0f0", 
                                       outline="#999999", width=2)
        
    def on_leave(self, event):
        self.draw_checkbox()
        
    def on_variable_change(self, *args):
        self.draw_checkbox()
        
    def pack(self, **kwargs):
        return self.frame.pack(**kwargs)
    
    def grid(self, **kwargs):
        return self.frame.grid(**kwargs)

def update_model():
    """Update the KNN model with selected features"""
    global scaler, knn, X_scaled, current_features
    
    # Get selected features
    selected_features = [f for f, var in feature_vars.items() if var.get()]
    
    if len(selected_features) < 2:
        messagebox.showwarning("Warning", "Please select at least 2 features for accurate recommendations")
        return False
    
    try:
        current_features = selected_features
        
        # Normalize features
        scaler = MinMaxScaler()
        X_scaled = scaler.fit_transform(df[current_features])
        
        # Fit KNN model
        knn = NearestNeighbors(n_neighbors=min(11, len(df)), metric='euclidean')
        knn.fit(X_scaled)
        
        # Update status with modern styling
        status_label.config(text=f"✅ Model updated with {len(current_features)} features", 
                           fg="#4CAF50")
        accuracy_label.config(text="🚀 Model ready - Click 'Get Recommendations' to see accuracy")
        
        print(f"Model updated with features: {current_features}")
        return True
        
    except Exception as e:
        messagebox.showerror("Error", f"Error updating model: {str(e)}")
        return False

def calculate_accuracy_metrics(sample_index, recommended_indices, distances):
    """Calculate various accuracy metrics for the recommendations"""
    sample_song = df.iloc[sample_index]
    recommended_songs = df.iloc[recommended_indices]
    
    metrics = {}
    
    # 1. Average distance (lower is better)
    metrics['avg_distance'] = np.mean(distances)
    metrics['min_distance'] = np.min(distances)
    metrics['max_distance'] = np.max(distances)
    
    # 2. Genre accuracy (if same genre)
    if 'track_genre' in df.columns:
        sample_genre = sample_song['track_genre']
        genre_matches = sum(1 for idx in recommended_indices 
                           if df.iloc[idx]['track_genre'] == sample_genre)
        metrics['genre_accuracy'] = (genre_matches / len(recommended_indices)) * 100
    
    # 3. Artist similarity (if same artist)
    sample_artist = sample_song['artists'].lower()
    artist_matches = sum(1 for idx in recommended_indices 
                        if sample_artist in df.iloc[idx]['artists'].lower())
    metrics['artist_similarity'] = (artist_matches / len(recommended_indices)) * 100
    
    # 4. Feature similarity scores for each feature
    sample_features = X_scaled[sample_index]
    feature_similarities = {}
    
    for i, feature in enumerate(current_features):
        sample_val = sample_features[i]
        recommended_vals = [X_scaled[idx][i] for idx in recommended_indices]
        
        # Calculate average absolute difference (lower is better)
        avg_diff = np.mean([abs(sample_val - val) for val in recommended_vals])
        similarity_score = max(0, (1 - avg_diff) * 100)  # Convert to percentage
        feature_similarities[feature] = similarity_score
    
    metrics['feature_similarities'] = feature_similarities
    
    # 5. Overall similarity score
    distance_score = max(0, (1 - metrics['avg_distance']) * 100)
    metrics['overall_similarity'] = distance_score
    
    return metrics

def recommend():
    """Generate song recommendations with accuracy metrics"""
    if knn is None:
        messagebox.showwarning("Warning", "Please update the model first by selecting features")
        return
        
    try:
        genre = genre_var.get().strip().lower()
        artist = artist_var.get().strip().lower()
        
        # Start with full dataset
        filtered_df = df.copy()
        
        # Track what filters were applied
        genre_filtered = False
        artist_filtered = False
        
        # Apply genre filter if specified
        if genre:
            filtered_df = filtered_df[filtered_df['track_genre'].str.lower() == genre]
            genre_filtered = True
            if filtered_df.empty:
                messagebox.showinfo("No Results", f"No songs found for genre: {genre}")
                return
        
        # Apply artist filter if specified
        if artist:
            filtered_df = filtered_df[filtered_df['artists'].str.lower().str.contains(artist, na=False)]
            artist_filtered = True
            if filtered_df.empty:
                messagebox.showinfo("No Results", f"No songs found for artist: {artist}")
                return
        
        if filtered_df.empty:
            messagebox.showinfo("No Results", "No songs found with the given filters.")
            return
        
        # Select a random song from filtered results
        sample_song = filtered_df.sample(1)
        sample_index = sample_song.index[0]
        
        # Find similar songs using KNN
        distances, indices = knn.kneighbors([X_scaled[sample_index]])
        recommended_indices = indices[0][1:]  # Exclude the sample song itself
        recommendation_distances = distances[0][1:]
        
        # Calculate accuracy metrics
        metrics = calculate_accuracy_metrics(sample_index, recommended_indices, recommendation_distances)
        
        # Get recommendation details
        recommendations = df.iloc[recommended_indices]
        
        # Group by track name to handle duplicates
        grouped = recommendations.groupby('track_name').agg({
            'artists': lambda x: ', '.join(set(x))
        }).reset_index()
        
        # Display results with enhanced formatting
        output.delete('1.0', tk.END)
        
        # Selected song info
        output.insert(tk.END, "🎵 SELECTED SONG:\n")
        output.insert(tk.END, f"Track: {sample_song['track_name'].iloc[0]}\n")
        output.insert(tk.END, f"Artist: {sample_song['artists'].iloc[0]}\n")
        if 'track_genre' in sample_song.columns:
            output.insert(tk.END, f"Genre: {sample_song['track_genre'].iloc[0]}\n")
        
        # Filter information
        if genre_filtered or artist_filtered:
            output.insert(tk.END, f"\n🔍 APPLIED FILTERS:\n")
            if genre_filtered:
                output.insert(tk.END, f"Genre Filter: {genre.title()}\n")
            if artist_filtered:
                output.insert(tk.END, f"Artist Filter: {artist.title()}\n")
        
        # Feature values of selected song
        output.insert(tk.END, f"\n📊 SELECTED SONG FEATURES:\n")
        for i, feature in enumerate(current_features):
            original_val = df.iloc[sample_index][feature]
            normalized_val = X_scaled[sample_index][i]
            output.insert(tk.END, f"{feature}: {original_val:.3f} (normalized: {normalized_val:.3f})\n")
        
        # Current settings
        output.insert(tk.END, f"\n📋 CURRENT SETTINGS:\n")
        output.insert(tk.END, f"Features Used: {', '.join(current_features)}\n")
        output.insert(tk.END, f"Total Features: {len(current_features)}\n")
        
        # Accuracy metrics
        output.insert(tk.END, f"\n🎯 ACCURACY METRICS:\n")
        output.insert(tk.END, f"Overall Similarity Score: {metrics['overall_similarity']:.1f}%\n")
        output.insert(tk.END, f"Average Distance: {metrics['avg_distance']:.4f}\n")
        
        if genre_filtered and 'genre_accuracy' in metrics:
            output.insert(tk.END, f"Genre Match Accuracy: {metrics['genre_accuracy']:.1f}%\n")
        
        if artist_filtered and metrics['artist_similarity'] > 0:
            output.insert(tk.END, f"Artist Similarity: {metrics['artist_similarity']:.1f}%\n")
        
        output.insert(tk.END, f"\n🔍 FEATURE SIMILARITY SCORES:\n")
        for feature, score in metrics['feature_similarities'].items():
            output.insert(tk.END, f"{feature}: {score:.1f}%\n")
        
        # Recommended songs
        output.insert(tk.END, f"\n🎧 RECOMMENDED SONGS:\n")
        output.insert(tk.END, "=" * 60 + "\n")
        
        for i, (idx, row) in enumerate(grouped.iterrows()):
            if i < len(recommendation_distances):
                output.insert(tk.END, f"♪ {row['track_name']}\n")
                output.insert(tk.END, f"  by {row['artists']}\n")
                output.insert(tk.END, f"  Distance: {recommendation_distances[i]:.4f}\n\n")
        
        # Update accuracy display
        accuracy_text = f"Overall: {metrics['overall_similarity']:.1f}%"
        if genre_filtered and 'genre_accuracy' in metrics:
            accuracy_text += f" | Genre: {metrics['genre_accuracy']:.1f}%"
        if artist_filtered and metrics['artist_similarity'] > 0:
            accuracy_text += f" | Artist: {metrics['artist_similarity']:.1f}%"
        accuracy_text += f" | Features: {len(current_features)}"
        
        accuracy_label.config(text=accuracy_text, fg="#2196F3")
            
    except Exception as e:
        messagebox.showerror("Error", f"An error occurred: {str(e)}")

def select_all_features():
    """Select all available features"""
    for var in feature_vars.values():
        var.set(True)

def deselect_all_features():
    """Deselect all features"""
    for var in feature_vars.values():
        var.set(False)

def apply_preset(preset_name):
    """Apply a preset feature selection"""
    presets = {
        'party': ['valence', 'energy', 'danceability', 'tempo'],
        'chill': ['valence', 'acousticness', 'energy', 'loudness', 'instrumentalness'],
        'workout': ['energy', 'tempo', 'danceability', 'loudness', 'valence'],
        'study': ['instrumentalness', 'acousticness', 'valence', 'energy'],
        'genre': ['acousticness', 'instrumentalness', 'speechiness', 'loudness', 'tempo'],
        'mood': ['valence', 'energy', 'danceability', 'acousticness', 'liveness'],
        'all': available_features
    }
    
    selected_features = presets.get(preset_name, available_features)
    
    # Clear all first
    for var in feature_vars.values():
        var.set(False)
    
    # Set selected features
    for feature, var in feature_vars.items():
        if feature in selected_features:
            var.set(True)

def reset_to_default():
    """Reset to balanced default selection"""
    apply_preset('mood')

def clear_filters():
    """Clear all input fields"""
    genre_var.set("")
    artist_var.set("")
    output.delete('1.0', tk.END)

def on_filter_change(*args):
    """Handle filter changes"""
    pass  # Placeholder for filter change handling

# GUI Setup with modern design and scrollbar
root = tk.Tk()
root.title("🎶 Advanced Song Recommendation System")
root.geometry("1300x900")
root.configure(bg='#f5f5f5')

# Create main container with scrollbar
container = tk.Frame(root)
canvas = tk.Canvas(container)
scrollbar = ttk.Scrollbar(container, orient="vertical", command=canvas.yview)
scrollable_frame = tk.Frame(canvas)

# Configure the canvas
scrollable_frame.bind("<Configure>",lambda e: canvas.configure(scrollregion=canvas.bbox("all"), width=e.width ))  # Make canvas width match scrollable_frame))

canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
canvas.configure(yscrollcommand=scrollbar.set)

# Pack everything
container.pack(fill="both", expand=True)
canvas.pack(side="left", fill="both", expand=True)
scrollbar.pack(side="right", fill="y")

# Configure modern styling
style = ttk.Style()
# Configure modern styling
style = ttk.Style()
style.theme_use('clam')

# Configure notebook styling
style.configure('TNotebook', background='#f5f5f5', borderwidth=0)
style.configure('TNotebook.Tab', 
               padding=[20, 10], 
               background='#e0e0e0',
               focuscolor='none')
style.map('TNotebook.Tab',
         background=[('selected', '#ffffff')],
         foreground=[('selected', '#333333')])

# Create main frame with modern styling inside the scrollable frame
main_frame = tk.Frame(scrollable_frame, bg='#f5f5f5')
main_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=20)

# Create modern header
header_frame = tk.Frame(main_frame, bg='#2196F3', height=80)
header_frame.pack(fill='x', pady=(0, 20))
header_frame.pack_propagate(False)

header_label = tk.Label(header_frame, 
                       text="🎶 Advanced Song Recommendation System",
                       font=('Arial', 18, 'bold'),
                       bg='#2196F3', fg='white')
header_label.pack(expand=True)

# Create notebook with enhanced styling
notebook = ttk.Notebook(main_frame)
notebook.pack(fill=tk.BOTH, expand=True)

# Tab 1: Feature Selection with modern design
feature_frame = tk.Frame(notebook, bg='#ffffff')
notebook.add(feature_frame, text="🔧 Feature Selection")

# Add padding frame
feature_content = tk.Frame(feature_frame, bg='#ffffff')
feature_content.pack(fill=tk.BOTH, expand=True, padx=20, pady=20)

# Title with modern styling
title_label = tk.Label(feature_content, 
                      text="Select Features for Recommendation",
                      font=('Arial', 16, 'bold'),
                      bg='#ffffff', fg='#333333')
title_label.pack(anchor='w', pady=(0, 20))

# Enhanced preset buttons with modern styling
preset_frame = tk.LabelFrame(feature_content, 
                           text="🎯 Quick Presets",
                           font=('Arial', 12, 'bold'),
                           bg='#ffffff', fg='#333333',
                           padx=15, pady=15)
preset_frame.pack(fill='x', pady=(0, 20))

# Create modern preset buttons
preset_buttons_frame = tk.Frame(preset_frame, bg='#ffffff')
preset_buttons_frame.pack(fill='x')

# Define enhanced button configurations
button_configs = [
    ("Party/Dancing", lambda: apply_preset('party'), "#FF6B6B", "#FF5252", "🎉"),
    ("Chill/Relaxing", lambda: apply_preset('chill'), "#4ECDC4", "#26A69A", "😌"),
    ("Workout", lambda: apply_preset('workout'), "#45B7D1", "#2196F3", "💪"),
    ("Study/Focus", lambda: apply_preset('study'), "#96CEB4", "#66BB6A", "📚"),
    ("Genre-Similar", lambda: apply_preset('genre'), "#FECA57", "#FFC107", "🎼"),
    ("Mood-Based", lambda: apply_preset('mood'), "#FF9FF3", "#E91E63", "🎭"),
    ("All Features", lambda: apply_preset('all'), "#A8E6CF", "#81C784", "🔧")
]

# Create modern buttons
for i, (text, command, bg_color, hover_color, emoji) in enumerate(button_configs):
    button = ModernButton(
        preset_buttons_frame, 
        text, 
        command, 
        bg_color, 
        hover_color, 
        text_color="white" if bg_color != "#FECA57" and bg_color != "#A8E6CF" else "black",
        emoji=emoji
    )
    button.pack(side='left', padx=5, pady=10, expand=True, fill='x')

# Enhanced feature selection with custom checkboxes
categories_frame = tk.LabelFrame(feature_content, 
                               text="🎵 Feature Categories",
                               font=('Arial', 12, 'bold'),
                               bg='#ffffff', fg='#333333',
                               padx=15, pady=15)
categories_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 20))

# Create feature checkboxes with modern styling
feature_vars = {}

# Define feature categories with descriptions
categories = {
    "🎭 Mood & Feel": {
        'features': ['valence', 'energy', 'danceability'],
        'color': '#FF9FF3'
    },
    "🎼 Sound Quality": {
        'features': ['acousticness', 'loudness', 'liveness'],
        'color': '#4ECDC4'
    },
    "🎤 Content Type": {
        'features': ['speechiness', 'instrumentalness'],
        'color': '#FECA57'
    },
    "⚡ Rhythm & Pace": {
        'features': ['tempo', 'time_signature', 'duration_ms'],
        'color': '#FF6B6B'
    },
    "🎹 Music Theory": {
        'features': ['key', 'mode'],
        'color': '#96CEB4'
    }
}

# Create category frames with enhanced styling
for i, (category, config) in enumerate(categories.items()):
    # Create modern category frame
    cat_frame = tk.Frame(categories_frame, bg=config['color'], relief='solid', borderwidth=1)
    cat_frame.grid(row=i//3, column=i%3, sticky='nsew', padx=8, pady=8)
    
    # Category header
    header = tk.Label(cat_frame, text=category, 
                     font=('Arial', 11, 'bold'),
                     bg=config['color'], fg='white' if config['color'] != '#FECA57' else 'black',
                     pady=8)
    header.pack(fill='x')
    
    # Features container
    features_container = tk.Frame(cat_frame, bg='#ffffff')
    features_container.pack(fill='both', expand=True, padx=2, pady=(0, 2))
    
    # Add custom checkboxes
    for j, feature in enumerate(config['features']):
        if feature in available_features:
            feature_vars[feature] = tk.BooleanVar(value=True)
            checkbox = CustomCheckbox(features_container, feature, 
                                    feature_vars[feature],
                                    check_color=config['color'])
            checkbox.pack(anchor='w', padx=8, pady=4)

# Configure grid weights
for i in range(2):
    categories_frame.grid_rowconfigure(i, weight=1)
for i in range(3):
    categories_frame.grid_columnconfigure(i, weight=1)

# Enhanced control buttons
control_frame = tk.Frame(feature_content, bg='#ffffff')
control_frame.pack(fill='x', pady=(0, 10))

# Left side buttons
left_buttons = tk.Frame(control_frame, bg='#ffffff')
left_buttons.pack(side='left')

select_all_btn = ModernButton(left_buttons, "Select All", select_all_features, 
                             "#4CAF50", "#45a049", emoji="✅")
select_all_btn.pack(side='left', padx=5)

deselect_all_btn = ModernButton(left_buttons, "Deselect All", deselect_all_features, 
                               "#f44336", "#da190b", emoji="❌")
deselect_all_btn.pack(side='left', padx=5)

default_btn = ModernButton(left_buttons, "Default", reset_to_default, 
                          "#FF9800", "#e68900", emoji="🔄")
default_btn.pack(side='left', padx=5)

# Right side update button
update_btn = ModernButton(control_frame, "Update Model", update_model, 
                         "#2196F3", "#1976D2", emoji="🚀")
update_btn.pack(side='right', padx=10)

# Enhanced status labels
status_label = tk.Label(feature_content, text="Select features and click 'Update Model'", 
                       font=('Arial', 11), bg='#ffffff', fg='#666666')
status_label.pack(pady=10)

# Tab 2: Recommendations with modern design
rec_frame = tk.Frame(notebook, bg='#ffffff')
notebook.add(rec_frame, text="🎵 Recommendations")

# Filter controls
filter_frame = ttk.Frame(rec_frame)
filter_frame.pack(fill='x', pady=5)

ttk.Label(filter_frame, text="Genre:").grid(row=0, column=0, sticky='e', padx=5)
genre_var = tk.StringVar()
genre_var.trace('w', on_filter_change)  # Track changes
genre_values = [""] + sorted(df['track_genre'].dropna().unique().tolist())
genre_entry = ttk.Combobox(filter_frame, textvariable=genre_var, values=genre_values, width=30)
genre_entry.grid(row=0, column=1, padx=5)

ttk.Label(filter_frame, text="Artist:").grid(row=0, column=2, sticky='e', padx=5)
artist_var = tk.StringVar()
artist_var.trace('w', on_filter_change)  # Track changes
artist_entry = ttk.Entry(filter_frame, textvariable=artist_var, width=30)
artist_entry.grid(row=0, column=3, padx=5)

# Control buttons
control_frame = ttk.Frame(rec_frame)
control_frame.pack(fill='x', pady=10)

recommend_btn = ModernButton(control_frame, "Get Recommendations", recommend, 
                           "#4CAF50", "#45a049", emoji="🎧")
recommend_btn.pack(side='left', padx=5)

clear_btn = ModernButton(control_frame, "Clear", clear_filters, 
                        "#f44336", "#da190b", emoji="🗑️")
clear_btn.pack(side='left', padx=5)

# Accuracy display
accuracy_label = tk.Label(control_frame, text="Select features and update model first", 
                         font=('Arial', 10, 'bold'), bg='#ffffff')
accuracy_label.pack(side='right', padx=5)

# Output text area with scrollbar
text_frame = ttk.Frame(rec_frame)
text_frame.pack(fill=tk.BOTH, expand=True, pady=10)

output = tk.Text(text_frame, wrap=tk.WORD, font=('Consolas', 9))
scrollbar = ttk.Scrollbar(text_frame, orient="vertical", command=output.yview)
output.configure(yscrollcommand=scrollbar.set)

output.pack(side='left', fill=tk.BOTH, expand=True)
scrollbar.pack(side='right', fill='y')

# Initialize with welcome message
output.insert(tk.END, "Welcome to Advanced Song Recommendation System! 🎵\n\n")
output.insert(tk.END, "🎯 Quick Start Guide:\n")
output.insert(tk.END, "1. Try a PRESET (Party, Chill, Workout, etc.) for instant setup\n")
output.insert(tk.END, "2. Or customize features by CATEGORY (Mood, Sound, Content, etc.)\n")
output.insert(tk.END, "3. Click 'Update Model' to apply your selection\n")
output.insert(tk.END, "4. Go to 'Recommendations' tab and get suggestions with accuracy!\n\n")
output.insert(tk.END, "💡 Preset Explanations:\n")
output.insert(tk.END, "🎉 Party: High energy, danceable tracks\n")
output.insert(tk.END, "😌 Chill: Relaxing, acoustic-friendly vibes\n")
output.insert(tk.END, "💪 Workout: High tempo, energetic beats\n")
output.insert(tk.END, "📚 Study: Instrumental, calm focus music\n")
output.insert(tk.END, "🎼 Genre: Similar musical style and production\n")
output.insert(tk.END, "🎭 Mood: Emotional similarity and live feel\n\n")
output.insert(tk.END, f"Dataset: {len(df):,} unique songs\n")
output.insert(tk.END, f"Genres: {len(df['track_genre'].unique())}\n")
output.insert(tk.END, f"Artists: {df['artists'].nunique():,}\n")
output.insert(tk.END, f"Available features: {len(available_features)}\n")

# Initialize model with default features
root.after(100, update_model)

if __name__ == "__main__":
    root.mainloop()

Dataset not found at dataset.csv
Please ensure the dataset file exists or update the path.
Error loading dataset: [Errno 2] No such file or directory: 'dataset.csv'


NameError: name 'df' is not defined